In [1]:
!pip install apache-beam

# Setup

First, you need to set up your environment, which includes installing `apache-beam` and downloading a text file from Cloud Storage to your local file system. We are using this file to test your pipeline.

In [2]:
def run(cmd):
  print('>> {}'.format(cmd))
  !{cmd}
  print('')

In [4]:
run('gsutil cat gs://dataflow-samples/shakespeare/tempest.txt')

>> gsutil cat gs://dataflow-samples/shakespeare/tempest.txt
	THE TEMPEST


	DRAMATIS PERSONAE


ALONSO	King of Naples.

SEBASTIAN	his brother.

PROSPERO	the right Duke of Milan.

ANTONIO	his brother, the usurping Duke of Milan.

FERDINAND	son to the King of Naples.

GONZALO	an honest old Counsellor.


ADRIAN	|
	|  Lords.
FRANCISCO	|


CALIBAN	a savage and deformed Slave.

TRINCULO	a Jester.

STEPHANO	a drunken Butler.

	Master of a Ship. (Master:)

	Boatswain. (Boatswain:)

	Mariners. (Mariners:)

MIRANDA	daughter to Prospero.

ARIEL	an airy Spirit.


IRIS	|
	|
CERES	|
	|
JUNO	|  presented by Spirits.
	|
Nymphs	|
	|
Reapers	|


	Other Spirits attending on Prospero.


SCENE	A ship at Sea: an island.




	THE TEMPEST


ACT I



SCENE I	On a ship at sea: a tempestuous noise
	of thunder and lightning heard.


	[Enter a Master and a Boatswain]

Master	Boatswain!

Boatswain	Here, master: what cheer?

Master	Good, speak to the mariners: fall to't, yarely,
	or we run ourselves aground: bestir,

In [5]:
# Run and print a shell command.
def run(cmd):
  print('>> {}'.format(cmd))
  !{cmd}
  print('')

# Install apache-beam.
# run('pip install --quiet apache-beam')

# Copy the input file into the local file system.
run('mkdir -p data')
run('gsutil cp gs://dataflow-samples/shakespeare/tempest.txt data/')

>> mkdir -p data

>> gsutil cp gs://dataflow-samples/shakespeare/tempest.txt data/
Copying gs://dataflow-samples/shakespeare/tempest.txt...
/ [1 files][ 97.0 KiB/ 97.0 KiB]                                                
Operation completed over 1 objects/97.0 KiB.                                     



# Word count with comments

Below is mostly the same code as above, but with comments explaining every line in more detail.

In [6]:
import apache_beam as beam
import re

inputs_pattern = 'data/*'
outputs_prefix = 'outputs/part'

# Running locally in the DirectRunner.
with beam.Pipeline() as pipeline:
  # Store the word counts in a PCollection.
  # Each element is a tuple of (word, count) of types (str, int).
  word_counts = (
      # The input PCollection is an empty pipeline.
      pipeline

      # Read lines from a text file.
      | 'Read lines' >> beam.io.ReadFromText(inputs_pattern)
      # Element type: str - text line

      # Use a regular expression to iterate over all words in the line.
      # FlatMap will yield an element for every element in an iterable.
      | 'Find words' >> beam.FlatMap(lambda line: re.findall(r"[a-zA-Z']+", line))
      # Element type: str - word

      # Create key-value pairs where the value is 1, this way we can group by
      # the same word while adding those 1s and get the counts for every word.
      | 'Pair words with 1' >> beam.Map(lambda word: (word, 1))
      # Element type: (str, int) - key: word, value: 1

      # Group by key while combining the value using the sum() function.
      | 'Group and sum' >> beam.CombinePerKey(sum)
      # Element type: (str, int) - key: word, value: counts
  )

  # We can process a PCollection through other pipelines too.
  (
      # The input PCollection is the word_counts created from the previous step.
      word_counts

      # Format the results into a string so we can write them to a file.
      | 'Format results' >> beam.Map(lambda word_count: str(word_count))
      # Element type: str - text line

      # Finally, write the results to a file.
      | 'Write results' >> beam.io.WriteToText(outputs_prefix)
  )

# Sample the first 20 results, remember there are no ordering guarantees.
run('head -n 200 {}-00000-of-*'.format(outputs_prefix))

>> head -n 200 outputs/part-00000-of-*
('THE', 11)
('TEMPEST', 11)
('DRAMATIS', 1)
('PERSONAE', 1)
('ALONSO', 48)
('King', 7)
('of', 271)
('Naples', 23)
('SEBASTIAN', 80)
('his', 95)
('brother', 16)
('PROSPERO', 129)
('the', 421)
('right', 4)
('Duke', 8)
('Milan', 22)
('ANTONIO', 69)
('usurping', 1)
('FERDINAND', 43)
('son', 20)
('to', 263)
('GONZALO', 58)
('an', 27)
('honest', 1)
('old', 6)
('Counsellor', 1)
('ADRIAN', 13)
('Lords', 1)
('FRANCISCO', 6)
('CALIBAN', 59)
('a', 266)
('savage', 2)
('and', 375)
('deformed', 1)
('Slave', 1)
('TRINCULO', 47)
('Jester', 1)
('STEPHANO', 67)
('drunken', 4)
('Butler', 1)
('Master', 7)
('Ship', 1)
('Boatswain', 18)
('Mariners', 5)
('MIRANDA', 58)
('daughter', 17)
('Prospero', 13)
('ARIEL', 72)
('airy', 2)
('Spirit', 3)
('IRIS', 6)
('CERES', 6)
('JUNO', 4)
('presented', 2)
('by', 61)
('Spirits', 6)
('Nymphs', 3)
('Reapers', 2)
('Other', 1)
('attending', 1)
('on', 58)
('SCENE', 10)
('A', 52)
('ship', 12)
('at', 47)
('Sea', 3)
('island', 22)
('ACT', 